# 程式與文件助手

# 選擇模型
請自由任意選擇下面兩個模型之一來進行範例，建議使用Gemini

## Gemini

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

API_KEY = "這邊請改成你自己的API_KEY值"
model_name = 'gemini-2.5-flash'

llm = ChatGoogleGenerativeAI(
    model=model_name,
    google_api_key=API_KEY
)

## LM Studio 或 Ollama

In [ ]:
from langchain_openai import ChatOpenAI
model_name = 'gemma-3-12b-it'  # 指定模型名稱，模型名稱會根據下載的模型不同而改變

# 根據使用LM Studio或Ollama來選擇適當的伺服器URL
base_url = 'http://localhost:1234/v1'  # LM Studio 本地伺服器的URL
# base_url = 'http://localhost:11434/v1' # Ollama 本地伺服器的URL

llm = ChatOpenAI(
    model=model_name,
    openai_api_key="not-needed",
    openai_api_base=base_url 
)

## 測試模型

In [2]:
from langchain_core.messages import HumanMessage
messages = [
    HumanMessage("簡短的說明機器學習的定義")
]
result = llm.invoke(messages)
print(result.content)

機器學習是一種人工智慧技術，讓電腦系統能夠從資料中「學習」，自動識別模式並做出預測或決策，而無需被明確地程式設計。


# 準備工具

In [15]:
# 專為 LLM 使用的工具集合（tools）
import os
from langchain_core.tools import tool
from pydantic import BaseModel, Field
import subprocess
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.document_loaders import UnstructuredWordDocumentLoader
from langchain_community.document_loaders import UnstructuredExcelLoader

# ------------------------------------------------------------
# 注意事項（給 LLM 與開發者）
# - 下列工具會在「工具可存取的根目錄」下執行所有檔案存取動作。
# - 所有函數在進行檔案/資料夾路徑操作時都會做絕對路徑檢查，防止跳脫根目錄。
# - 工具回傳文字或錯誤字串（錯誤會以中文訊息回報），以便 LLM 判斷下一步。
# - 工具設計時假定輸入是由 LLM 產生的參數（例如：檔名、子資料夾名稱），因此在 docstring 中用簡潔且結構化的格式說明用途與限制。
# ------------------------------------------------------------

# 設定（程式內變數）
ROOT_DIR = "C:/DATA/TEST"

# -------------------------------
# 型別定義（Pydantic schema）
# -------------------------------
class file_list_input(BaseModel):
    relative_path: str = Field(description="子資料夾名稱。可為空字串，代表根目錄。")

class read_txt_file_input(BaseModel):
    filename: str = Field(description="要讀取的文字檔相對路徑（相對於工具可存取的根目錄）。")

class save_txt_file_input(BaseModel):
    filename: str = Field(description="要儲存的文字檔相對路徑（相對於工具可存取的根目錄）。")
    content: str = Field(description="要寫入的文字內容。")

class run_python_input(BaseModel):
    filename: str = Field(description="要執行的 Python 檔案相對路徑（相對於工具可存取的根目錄）。")

class read_pdf_file_input(BaseModel):
    filename: str = Field(description="要讀取的 PDF 檔相對路徑（相對於工具可存取的根目錄）。")

class read_docx_file_input(BaseModel):
    filename: str = Field(description="要讀取的 Word (.docx) 檔相對路徑（相對於工具可存取的根目錄）。")

class read_xlsx_file_input(BaseModel):
    filename: str = Field(description="要讀取的 Excel (.xlsx) 檔相對路徑（相對於工具可存取的根目錄）。")

# ------------------------------------------------------------
# 協助函式
# ------------------------------------------------------------

def _abs_and_check(path: str) -> (str, str):
    """
    取得絕對路徑並檢查是否仍在工具可存取的根目錄內。

    Args:
        path: 相對路徑或完整路徑（相對於工具可存取的根目錄）。

    Returns:
        (abs_path, abs_root): 回傳絕對路徑與根目錄的絕對路徑。

    Notes:
        - 這個 helper 只負責計算與回傳，實際的允許/拒絕由呼叫端處理，以維持錯誤訊息的一致性。
    """
    abs_path = os.path.abspath(os.path.join(ROOT_DIR, path))
    abs_root = os.path.abspath(ROOT_DIR)
    return abs_path, abs_root


def document_to_txt(documents):
    """
    將文件 loader 回傳的多個 Document 物件合併成純文字。

    Args:
        documents: 一個可疊代的物件，每個元素具有 .page_content 屬性。

    Returns:
        字串：以換行符號連接每一頁的內容。
    """
    return "".join([doc.page_content for doc in documents])

# ==============================================================
# TOOL: 列出指定子資料夾的檔案
# ==============================================================

@tool("file_list", args_schema=file_list_input)
def file_list(relative_path=""):
    """
    列出「工具可存取的根目錄/relative_path」底下的檔案與子資料夾清單。

    Args:
        relative_path (str): 子資料夾相對路徑；空字串代表根目錄。

    Returns:
        dict:
            {
                "files": [檔案名稱...],
                "folders": [子資料夾名稱...]
            }
        或錯誤訊息字串。

    Error behavior:
        - 若路徑不在工具可存取的根目錄內，回傳錯誤字串。
        - 若資料夾不存在或無法存取，回傳對應的錯誤字串。

    安全限制：
        僅能列出根目錄底下的內容，禁止任何試圖跳脫根目錄的請求。
    """

    # 1) 取得絕對路徑（並透過 _abs_and_check 檢查合法性）
    abs_folder, abs_root = _abs_and_check(relative_path)

    # 2) 路徑安全檢查：禁止跳脫根目錄
    if not abs_folder.startswith(abs_root):
        return f"錯誤：不能存取超出根目錄的路徑！({relative_path})"

    # 3) 列出檔案與資料夾
    try:
        items = os.listdir(abs_folder)
        files = [
            f for f in items
            if os.path.isfile(os.path.join(abs_folder, f))
        ]
        folders = [
            d for d in items
            if os.path.isdir(os.path.join(abs_folder, d))
        ]

        return {
            "files": files,
            "folders": folders
        }

    except FileNotFoundError:
        return f"找不到資料夾: {abs_folder}"
    except PermissionError:
        return f"沒有權限讀取資料夾: {abs_folder}"
# ==============================================================
# TOOL: 讀取文字檔內容
# ==============================================================

@tool("read_txt_file", args_schema=read_txt_file_input)
def read_txt_file(filename):
    """
    讀取文字檔內容，並以 UTF-8 解碼後回傳整個文字內容。

    Args:
        filename (str): 相對路徑（相對於工具可存取的根目錄）。

    Returns:
        str: 檔案內容，或錯誤訊息字串。

    Error behavior:
        - 若嘗試跳脫根目錄，回傳錯誤字串。
        - 找不到檔案、權限問題或編碼錯誤，皆回傳對應的錯誤字串。

    安全限制：
        僅允許在工具可存取的根目錄下讀取檔案。
    """

    abs_file, abs_root = _abs_and_check(filename)

    if not abs_file.startswith(abs_root):
        return f"錯誤：不能存取超出根目錄的檔案！({filename})"

    try:
        # 明確指定 UTF-8，遇到不同編碼會回傳清楚錯誤訊息
        with open(abs_file, "r", encoding="utf-8") as f:
            return f.read()
    except FileNotFoundError:
        return f"找不到檔案: {abs_file}"
    except PermissionError:
        return f"沒有權限讀取檔案: {abs_file}"
    except UnicodeDecodeError:
        return f"檔案編碼錯誤，無法以 UTF-8 讀取: {abs_file}"

# ==============================================================
# TOOL: 儲存文字檔
# ==============================================================

@tool("save_txt_file", args_schema=save_txt_file_input)
def save_txt_file(filename, content):
    """
    將文字內容寫入指定檔案（覆蓋既有檔案），若父資料夾不存在則自動建立。

    Args:
        filename (str): 相對路徑（相對於工具可存取的根目錄）。
        content (str): 要寫入的文字內容。

    Returns:
        str: 成功訊息或錯誤訊息字串。

    Error behavior:
        - 若嘗試跳脫根目錄或無寫入權限，回傳錯誤字串。
        - 其他檔案系統錯誤會以 OSError 的字串形式回傳。
    """

    abs_file, abs_root = _abs_and_check(filename)

    if not abs_file.startswith(abs_root):
        return f"錯誤：不能存取超出根目錄的檔案！({filename})"

    # 確保父資料夾存在
    os.makedirs(os.path.dirname(abs_file), exist_ok=True)

    try:
        with open(abs_file, "w", encoding="utf-8") as f:
            f.write(content)
        return f"成功儲存檔案: {abs_file}"
    except PermissionError:
        return f"沒有權限寫入檔案: {abs_file}"
    except OSError as e:
        return f"寫入檔案時發生錯誤: {e}"

# ==============================================================
# TOOL: 執行 Python 程式
# ==============================================================

@tool("run_python", args_schema=run_python_input)
def run_python(filename):
    """
    在工具可存取的根目錄下執行指定的 Python 腳本，並回傳 stdout 或 stderr 內容。

    Args:
        filename (str): 要執行的 Python 檔案相對路徑（相對於工具可存取的根目錄）。

    Returns:
        str: 執行成功時回傳 stdout；執行失敗時回傳 stderr；或發生例外則回傳例外說明。

    Security / Behavior:
        - 執行時會把工作目錄設為工具可存取的根目錄，避免腳本在其他路徑執行。
        - 不會在此處注入額外的環境變數或參數（簡潔、安全）。
    """

    abs_file, abs_root = _abs_and_check(filename)

    if not abs_file.startswith(abs_root):
        return f"錯誤：不能存取超出根目錄的檔案！({filename})"

    # 確保父資料夾存在（若不存在，subprocess 也會失敗；此處先建立以避免部分錯誤）
    os.makedirs(os.path.dirname(abs_file), exist_ok=True)

    try:
        # 使用系統 python 呼叫指定檔案，並在工具根目錄執行
        result = subprocess.run([
            "python", abs_file
        ], capture_output=True, text=True, cwd=abs_root)

        # 根據 returncode 回傳相對應內容（stdout 或 stderr）
        if result.returncode == 0:
            return result.stdout
        else:
            return result.stderr

    except Exception as e:
        # 以簡潔的錯誤格式回傳，方便 LLM 判讀
        return f"執行錯誤：{type(e).__name__} - {e}"

# ==============================================================
# TOOL: 讀取 PDF
# ==============================================================

@tool("read_pdf_file", args_schema=read_pdf_file_input)
def read_pdf_file(filename):
    """
    讀取 PDF 並將內容回傳為純文字。

    Args:
        filename (str): PDF 檔案相對路徑（相對於工具可存取的根目錄）。

    Returns:
        str: 合併後的純文字內容，或錯誤訊息字串。

    Error behavior:
        - 檔案不存在、權限不足或路徑安全檢查失敗時會回傳對應的錯誤訊息。
    """

    abs_file, abs_root = _abs_and_check(filename)

    if not abs_file.startswith(abs_root):
        return f"錯誤：不能存取超出根目錄的檔案！({filename})"

    try:
        loader = PyPDFLoader(abs_file)
        documents = loader.load()
        return document_to_txt(documents)
    except FileNotFoundError:
        return f"找不到檔案: {abs_file}"
    except PermissionError:
        return f"沒有權限讀取檔案: {abs_file}"

# ==============================================================
# TOOL: 讀取 Word (.docx)
# ==============================================================

@tool("read_docx_file", args_schema=read_docx_file_input)
def read_docx_file(filename):
    """
    讀取 .docx 文件，並回傳純文字內容。

    Args:
        filename (str): .docx 檔案相對路徑（相對於工具可存取的根目錄）。

    Returns:
        str: 合併後的純文字內容，或錯誤訊息字串。
    """

    abs_file, abs_root = _abs_and_check(filename)

    if not abs_file.startswith(abs_root):
        return f"錯誤：不能存取超出根目錄的檔案！({filename})"

    try:
        loader = UnstructuredWordDocumentLoader(abs_file)
        documents = loader.load()
        return document_to_txt(documents)
    except FileNotFoundError:
        return f"找不到檔案: {abs_file}"
    except PermissionError:
        return f"沒有權限讀取檔案: {abs_file}"

# ==============================================================
# TOOL: 讀取 Excel (.xlsx)
# ==============================================================

@tool("read_xlsx_file", args_schema=read_xlsx_file_input)
def read_xlsx_file(filename):
    """
    讀取 .xlsx 文件，並回傳純文字內容。

    Args:
        filename (str): .xlsx 檔案相對路徑（相對於工具可存取的根目錄）。

    Returns:
        str: 合併後的純文字內容，或錯誤訊息字串。
    """

    abs_file, abs_root = _abs_and_check(filename)

    if not abs_file.startswith(abs_root):
        return f"錯誤：不能存取超出根目錄的檔案！({filename})"

    try:
        loader = UnstructuredExcelLoader(abs_file)
        documents = loader.load()
        return document_to_txt(documents)
    except FileNotFoundError:
        return f"找不到檔案: {abs_file}"
    except PermissionError:
        return f"沒有權限讀取檔案: {abs_file}"

# End of file


## 工具功能測試

In [27]:
# 列出工作目錄的檔案
file_list.invoke({"relative_path":""})

{'files': ['CV.docx',
  'CV.html',
  'CV.pdf',
  'CV.txt',
  'heart.csv',
  'Iris.csv',
  'Iris.xlsx'],
 'folders': []}

In [21]:
# 測試能不能突破安全限制
file_list.invoke({"relative_path":"../test"})

'錯誤：不能存取超出根目錄的路徑！(../test)'

In [22]:
# 測試能不能突破安全限制
file_list.invoke({"relative_path":"C:/"})

'錯誤：不能存取超出根目錄的路徑！(C:/)'

In [23]:
# 測試EXCEL檔案的讀取能力
read_xlsx_file.invoke({"filename":"Iris.xlsx"})

'Id 花萼長度 花萼寬度 花瓣長度 花瓣寬度 類別 1 5.1 3.5 1.4 0.2 山鳶尾 2 4.9 3 1.4 0.2 山鳶尾 3 4.7 3.2 1.3 0.2 山鳶尾 4 4.6 3.1 1.5 0.2 山鳶尾 5 5 3.6 1.4 0.2 山鳶尾 6 5.4 3.9 1.7 0.4 山鳶尾 7 4.6 3.4 1.4 0.3 山鳶尾 8 5 3.4 1.5 0.2 山鳶尾 9 4.4 2.9 1.4 0.2 山鳶尾 10 4.9 3.1 1.5 0.1 山鳶尾 11 5.4 3.7 1.5 0.2 山鳶尾 12 4.8 3.4 1.6 0.2 山鳶尾 13 4.8 3 1.4 0.1 山鳶尾 14 4.3 3 1.1 0.1 山鳶尾 15 5.8 4 1.2 0.2 山鳶尾 16 5.7 4.4 1.5 0.4 山鳶尾 17 5.4 3.9 1.3 0.4 山鳶尾 18 5.1 3.5 1.4 0.3 山鳶尾 19 5.7 3.8 1.7 0.3 山鳶尾 20 5.1 3.8 1.5 0.3 山鳶尾 21 5.4 3.4 1.7 0.2 山鳶尾 22 5.1 3.7 1.5 0.4 山鳶尾 23 4.6 3.6 1 0.2 山鳶尾 24 5.1 3.3 1.7 0.5 山鳶尾 25 4.8 3.4 1.9 0.2 山鳶尾 26 5 3 1.6 0.2 山鳶尾 27 5 3.4 1.6 0.4 山鳶尾 28 5.2 3.5 1.5 0.2 山鳶尾 29 5.2 3.4 1.4 0.2 山鳶尾 30 4.7 3.2 1.6 0.2 山鳶尾 31 4.8 3.1 1.6 0.2 山鳶尾 32 5.4 3.4 1.5 0.4 山鳶尾 33 5.2 4.1 1.5 0.1 山鳶尾 34 5.5 4.2 1.4 0.2 山鳶尾 35 4.9 3.1 1.5 0.1 山鳶尾 36 5 3.2 1.2 0.2 山鳶尾 37 5.5 3.5 1.3 0.2 山鳶尾 38 4.9 3.1 1.5 0.1 山鳶尾 39 4.4 3 1.3 0.2 山鳶尾 40 5.1 3.4 1.5 0.2 山鳶尾 41 5 3.5 1.3 0.3 山鳶尾 42 4.5 2.3 1.3 0.3 山鳶尾 43 4.4 3.2 1.3 0.2 山鳶尾 44 5 3.5 1.6 0.6 山鳶

In [24]:
# 測試Word檔案的讀取能力
read_docx_file.invoke({"filename":"CV.docx"})

'姓名：陳仁政\n\n出生年份：1972\n\n學術與專業背景：\n\n博士學位｜中原大學電子研究所\n\n博士後研究｜中央研究院資訊科學研究所\n\n產業與教育經歷：\n\n資深工程師｜吉鴻電子\n\n正工程師｜冠捷科技\n\n資料科學家｜104人力銀行人資學院\n\n兼任實務教師｜長庚大學工商管理系\n\n顧問｜104人力銀行人資學院\n\n現職：\n\n講師｜台灣人工智能產業協會\n\n講師｜實踐大學推廣中心\n\n講師｜緯育TibaMe\n\n專案經驗：\n\n智慧金融｜收鈔機韌體開發（偽鈔偵測）\n\n消費電子｜電視韌體開發\n\n智慧交通與監控\n\n\n\n高承載管制違規偵測系統\n\n手持式雷射測距儀\n\n區間測速系統\n\n違停偵測系統\n\n活動地磅系統\n\n測速照相系統\n\nAI與人力資源分析｜人才適任與久任度評估系統\n\n專長領域：\n\n人工智慧與機器學習｜機器學習、生成式 AI、大語言模型（LLM）\n\n電腦視覺與影像處理｜影像識別、智慧監控系統開發\n\n嵌入式系統與數位電路｜數位電路設計、嵌入式系統開發\n\n軟體開發與工程｜程式設計、多平台軟硬體整合\n\n課程資訊：\n\n實踐大學推廣部\xa0人工智慧與大數據應用全修班\xa02020/08\n\n華岡興業基金會\xa0AI人力資源大數據分析課程\xa02021/07\n\n實踐大學推廣部\xa0Python與資料處理實戰訓練班\xa02021/11\n\n金融研訓院\xa0AI與大數據在HR專業實務的應用workshop\xa02022/04\n\nAutoML AI人工智慧科技數據分析人才培育暨認證師資培訓教師研習會\xa02022/07 (東海大學、元智大學、世新大學、真理大學、長庚大學)\n\n長庚大學工商系碩士班\xa0商業資料科學實務與應用\xa02022\n\n中華郵政AI Workshop 2022/11\n\n長庚醫院 AutoML Workshop\xa02023-04\n\n\n\n南亞科技 ChatGPT技術原理及應用 2023-07\n\n長庚醫院 AI計畫撰寫工作坊\xa02023-07\n\n人工智慧產業協會\xa0AI人力資源大數據分析課程\xa02023/07\n\n實踐大學推廣部 AI影像辨識工程師培訓班 2024/06\n\n金屬工

In [25]:
# 測試PDF檔案讀取能力
read_pdf_file.invoke({"filename":"CV.pdf"})

'姓名：陳仁政 \n \n出生年份：1972 \n \n學術與專業背景： \n\uf0a7 博士學位｜中原大學電子研究所 \n\uf0a7 博士後研究｜中央研究院資訊科學研究所 \n \n產業與教育經歷： \n\uf0a7 資深工程師｜吉鴻電子 \n\uf0a7 正工程師｜冠捷科技 \n\uf0a7 資料科學家｜104人力銀行人資學院 \n\uf0a7 兼任實務教師｜長庚大學工商管理系 \n\uf0a7 顧問｜104人力銀行人資學院 \n \n現職： \n\uf0a7 講師｜台灣人工智能產業協會 \n\uf0a7 講師｜實踐大學推廣中心 \n\uf0a7 講師｜緯育TibaMe \n \n專案經驗： \n\uf0a7 智慧金融｜收鈔機韌體開發（偽鈔偵測） \n\uf0a7 消費電子｜電視韌體開發 \n\uf0a7 智慧交通與監控\uf0a7 高承載管制違規偵測系統 \n\uf0a7 手持式雷射測距儀 \n\uf0a7 區間測速系統 \n\uf0a7 違停偵測系統 \n\uf0a7 活動地磅系統 \n\uf0a7 測速照相系統 \n\uf0a7 AI與人力資源分析｜人才適任與久任度評估系統 \n \n專長領域： \n\uf0a7 人工智慧與機器學習｜機器學習、生成式 AI、大語言模型（LLM） \n\uf0a7 電腦視覺與影像處理｜影像識別、智慧監控系統開發 \n\uf0a7 嵌入式系統與數位電路｜數位電路設計、嵌入式系統開發 \n\uf0a7 軟體開發與工程｜程式設計、多平台軟硬體整合 \n \n課程資訊： \n\uf0a7 實踐大學推廣部 人工智慧與大數據應用全修班 2020/08 \n\uf0a7 華岡興業基金會 AI人力資源大數據分析課程 2021/07 \n\uf0a7 實踐大學推廣部 Python與資料處理實戰訓練班 2021/11 \n\uf0a7 金融研訓院 AI與大數據在HR專業實務的應用workshop 2022/04 \n\uf0a7 AutoML AI人工智慧科技數據分析人才培育暨認證師資培訓教師研習\n會 2022/07 (東海大學、元智大學、世新大學、真理大學、長庚大學) \n\uf0a7 長庚大學工商系碩士班 商業資料科學實務與應用 2022 \n\uf0a7 中華郵政AI Workshop 2022/11 \n\uf0a7 長庚醫院 A

In [26]:
# 再次測試安全限制
read_pdf_file.invoke({"filename":"../CV.pdf"})

'錯誤：不能存取超出根目錄的檔案！(../CV.pdf)'

In [30]:
# 寫入文字檔的能力
save_txt_file.invoke({"filename":"test.py", "content":"print('Hello World!')"})

'成功儲存檔案: C:\\DATA\\TEST\\test.py'

In [31]:
# 執行pyton程式的能力
run_python.invoke({"filename":"test.py"})

'Hello World!\n'

# 將工具接入對話機器人

In [ ]:
from langchain_core.messages import ToolMessage, HumanMessage, SystemMessage, AIMessageChunk
from langchain_core.output_parsers import StrOutputParser

class stream_chat_bot:
    def __init__(self, llm, tools):
        # 初始化對話機器人，傳入 LLM 與可用工具列表
        self.tools = tools
        # 將 LLM 綁定（bind）工具，使其具備自動呼叫工具的能力
        self.llm_with_tools = llm.bind_tools(tools)

        # 系統提示詞（System Prompt），用來設定 LLM 的角色與行為
        system_prompt = '''
你是一位智慧型個人助理，能夠根據使用者的問題主動判斷是否需要使用工具。
請以清楚、簡潔的方式回答問題。
若問題需要外部資料，請直接使用可用的工具完成查詢，不需向使用者確認。
'''        
        # 初始化訊息列表，第一條訊息是系統指令
        self.message = [SystemMessage(system_prompt)]

        # 將 LLM 的回應解析為純文字格式的工具
        self.str_parser = StrOutputParser()
       
    def chat_generator(self, text):
        """
        主對話生成函式（生成器形式）。
        逐步執行 LLM 回應與工具調用，並即時回傳每一步的結果。
        """
        # 將使用者的輸入加入訊息列表
        self.message.append(HumanMessage(text))        
        
        while True:
            # 呼叫 LLM，傳入完整訊息歷史
            
            final_ai_message = AIMessageChunk(content="")
            for chunk in self.llm_with_tools.stream(self.message):
                final_ai_message += chunk
                if hasattr(chunk, 'content') and chunk.content:
                    yield self.str_parser.invoke(chunk)
            
            response = final_ai_message
            
            # 將 LLM 回應加入訊息列表
            self.message.append(response)

            # 檢查 LLM 是否要求呼叫工具
            is_tools_call = False
            for tool_call in response.tool_calls:
                is_tools_call = True

                # 顯示 LLM 要執行的工具名稱與參數
                msg = f'[執行]: {tool_call["name"]}({tool_call["args"]})\n\n' #完整訊息
                # msg = f'[執行]: {tool_call["name"]}()\n\n' #簡易訊息
                yield msg  # 使用 yield 讓結果能即時顯示在輸出中

                # 實際執行工具（根據工具名稱動態呼叫對應物件）
                tool_result = globals()[tool_call['name']].invoke(tool_call['args']) 

                # 顯示工具執行結果
                # msg = f'[結果]: {tool_result}\n\n'
                # yield msg

                # 將工具執行結果封裝成 ToolMessage 回傳給 LLM
                tool_message = ToolMessage(
                    content=str(tool_result),          # 工具執行的文字結果
                    name=tool_call["name"],            # 工具名稱
                    tool_call_id=tool_call["id"],      # 工具呼叫 ID（讓 LLM 知道對應哪個呼叫）
                )
                # 將工具回傳結果加入訊息列表，提供 LLM 下一輪參考
                self.message.append(tool_message)
            
            # 若這一輪沒有任何工具呼叫，表示 LLM 已經生成最終回覆
            if is_tools_call == False:
                # 將 LLM 回應解析成純文字並輸出
                # yield self.str_parser.invoke(response)
                return  # 結束對話流程

    def chat(self, text, print_output=False):
        """
        封裝版對話函式。
        會收集 chat_generator 的所有輸出，並組合成完整的回覆字串。
        """
        msg = ''
        # 逐步取得 chat_generator 的產出內容
        for chunk in self.chat_generator(text):
            msg += f"{chunk}"
            if print_output:
                print(chunk, end='')
        # 回傳最終組合的對話內容
        return msg


In [16]:
tools = [
    file_list,
    read_txt_file,
    save_txt_file,
    run_python,
    read_pdf_file,
    read_docx_file,
    read_xlsx_file,
]

## 測試：文件內容分析

In [45]:
bot = stream_chat_bot(llm, tools)

In [47]:
response = bot.chat("工作區有哪些檔案？", True)

[執行]: file_list()

您的工作區包含以下檔案：'CV.docx', 'CV.html', 'CV.pdf', 'CV.txt', 'heart.csv', 'Iris.csv', 'Iris.xlsx'。

In [48]:
response = bot.chat("CV.pdf內容跟什麼有關？", True)

[執行]: read_pdf_file()

CV.pdf 的內容是一份名為陳仁政的履歷。內容包含：

*   **個人基本資料**：姓名、出生年份。
*   **學術與專業背景**：中原大學電子研究所博士、中央研究院資訊科學研究所博士後研究。
*   **產業與教育經歷**：資深工程師、正工程師、資料科學家、兼任實務教師、顧問。
*   **現職**：台灣人工智能產業協會、實踐大學推廣中心、緯育TibaMe 的講師。
*   **專案經驗**：智慧金融（收鈔機韌體開發）、消費電子（電視韌體開發）、智慧交通與監控（多種偵測系統）、AI 與人力資源分析（人才適任與久任度評估系統）。
*   **專長領域**：人工智慧與機器學習、電腦視覺與影像處理、嵌入式系統與數位電路、軟體開發與工程。
*   **課程資訊**：列出了他教授或參與的關於人工智慧、大數據、Python、人力資源分析、電腦視覺和大型語言模型的課程和工作坊。

In [50]:
response = bot.chat("他有土木相關的專長嗎？", True)

根據 CV 內容，陳仁政先生的專長領域主要在人工智慧與機器學習、電腦視覺與影像處理、嵌入式系統與數位電路、軟體開發與工程，並且專案經驗也多與這些領域相關，例如智慧金融、消費電子、智慧交通與監控系統以及 AI 與人力資源分析。

CV 中**沒有提到**與土木工程相關的專長或經驗。

In [51]:
response = bot.chat("給我一份他的簡介摘要，讓大家能夠快速認識他", True)

陳仁政先生是一位在人工智慧、機器學習、電腦視覺和軟體開發領域經驗豐富的專家。他擁有中原大學電子研究所博士學位，並在中研院資訊科學所進行博士後研究。

在職涯方面，他曾擔任資深工程師、資料科學家，目前是台灣人工智能產業協會、實踐大學推廣中心及緯育TibaMe 的講師。他參與過多項專案，涵蓋智慧金融、消費電子、智慧交通監控系統以及 AI 人力資源分析。他的專長領域包含人工智慧、機器學習、大語言模型、電腦視覺、嵌入式系統和多平台軟硬體整合。

# 測試：程式與資料分析助手

In [76]:
bot = stream_chat_bot(llm, tools)

In [77]:
response = bot.chat("heart.csv是什麼資料?", True)

[執行]: read_txt_file({'filename': 'heart.csv'})

這個 `heart.csv` 檔案看起來是一個包含心臟疾病相關數據的 CSV (逗號分隔值) 檔案。

檔案的標頭列 (第一行) 顯示了以下欄位：

*   **age**: 年齡
*   **sex**: 性別 (1 = 男性, 0 = 女性)
*   **cp**: 胸痛類型 (chest pain type)
*   **trestbps**: 靜息血壓 (resting blood pressure)
*   **chol**: 血清膽固醇 (serum cholestoral)
*   **fbs**: 空腹血糖 (fasting blood sugar > 120 mg/dl)
*   **restecg**: 靜息心電圖結果 (resting electrocardiographic results)
*   **thalach**: 最大心率 (maximum heart rate achieved)
*   **exang**: 運動誘發心絞痛 (exercise induced angina)
*   **oldpeak**: 運動引起的 ST 段下降 (ST depression induced by exercise relative to rest)
*   **slope**: 運動高峰期 ST 段的斜率 (the slope of the peak exercise ST segment)
*   **ca**: 主要血管數量 (number of major vessels (0-3) colored by flourosopy)
*   **thal**: 地中海貧血類型 (thalassemia)
*   **target**: 預測目標 (1 = 有心臟病, 0 = 無心臟病)

這些資料通常用於建立預測模型，判斷一個人是否患有心臟疾病。

In [78]:
response = bot.chat("那些欄位跟心臟病有高相關性？", True)

[執行]: save_txt_file({'content': '\nimport pandas as pd\n\ntry:\n    df = pd.read_csv("heart.csv")\n    correlations = df.corr()[\'target\'].sort_values(ascending=False)\n    print("與心臟病 (target) 欄位相關性最高的欄位：")\n    print(correlations)\nexcept FileNotFoundError:\n    print("錯誤：heart.csv 檔案未找到。")\nexcept Exception as e:\n    print(f"處理檔案時發生錯誤: {e}")\n', 'filename': 'analyze_heart_data.py'})

[執行]: run_python({'filename': 'analyze_heart_data.py'})

根據相關性分析結果，與心臟病 (target) 欄位相關性最高的欄位 (絕對值越大，相關性越高) 如下：

**正相關 (值越高，心臟病風險越高)：**

*   **cp (胸痛類型)**: 0.433798
*   **thalach (最大心率)**: 0.421741
*   **slope (運動高峰期 ST 段的斜率)**: 0.345877

**負相關 (值越高，心臟病風險越低；或值越低，心臟病風險越高)：**

*   **exang (運動誘發心絞痛)**: -0.436757
*   **oldpeak (運動引起的 ST 段下降)**: -0.430696
*   **ca (主要血管數量)**: -0.391724
*   **thal (地中海貧血類型)**: -0.344029
*   **sex (性別)**: -0.280937 (這表示男性 (1) 相較於女性 (0) 患心臟病的機率較高，因為性別為 0 時，target 相關性會相對較高)
*   **age (年齡)**: -0.225439 (這可能表示在某些年齡段，心臟病風險的變化趨勢)
*   **trestbps (靜息血壓)**: -0.144931
*   **chol (血清膽固醇)

# 測試：資料分析與圖表助手

In [85]:
bot = stream_chat_bot(llm, tools)

In [86]:
response = bot.chat("Iris.csv有那些欄位？", True)

[執行]: read_txt_file({'filename': 'Iris.csv'})

Iris.csv 檔案包含以下欄位：

*   Id
*   花萼長度
*   花萼寬度
*   花瓣長度
*   花瓣寬度
*   類別

In [87]:
response = bot.chat("使用plotly畫出花萼寬度與花瓣長度的散佈圖，並用類別來當作顏色區分", True)

[執行]: save_txt_file({'content': '\nimport pandas as pd\nimport plotly.express as px\n\n# Read the CSV file\ndf = pd.read_csv("Iris.csv")\n\n# Create the scatter plot\nfig = px.scatter(df, x="花萼寬度", y="花瓣長度", color="類別",\n                 title="花萼寬度 vs 花瓣長度 (依類別區分)")\n\n# Save the plot to an HTML file\nfig.write_html("iris_scatter_plot.html")\n\nprint("散佈圖已儲存至 iris_scatter_plot.html")\n', 'filename': 'plot_iris.py'})

[執行]: run_python({'filename': 'plot_iris.py'})

散佈圖已成功儲存至 `iris_scatter_plot.html` 檔案中。您可以開啟該檔案查看花萼寬度與花瓣長度的散佈圖，並以類別進行顏色區分。

# 接入Gradio介面

In [17]:
import gradio as gr

# 建立 chat_bot 物件，並將 LLM 以及兩個工具（get_coordinates, get_weather）傳入
# 這樣 LLM 在回答時就能自動選擇使用這些工具
bot = stream_chat_bot(llm, tools)

# 定義一個用於 Gradio 聊天介面的函式
# message：使用者輸入的訊息
# history：對話歷史（Gradio 會自動傳入）
def chat_function(message, history):
    partial_response = ""  # 用來累積 LLM 的回應文字
    # chat_generator 是一個生成器 (generator)，會逐步產生模型或工具執行的輸出
    for chunk in bot.chat_generator(message):
        partial_response += f'{chunk}'  # 將每個輸出逐步串起來
        yield partial_response  # 即時回傳當前的部分結果，讓介面可以即時顯示

# 建立 Gradio 的聊天介面
# - chat_function：處理每次使用者輸入的函式
demo = gr.ChatInterface(chat_function)

# 主程式進入點
# 啟動 Gradio Web 介面
if __name__ == "__main__":
    demo.launch()


* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.
